In [1]:
# Стандартные библиотеки Python
import json
import os
import time

from datetime import datetime
from typing import List, Dict, Any

# Внешние библиотеки
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import clear_output

# Локальные модули
from board import Board, Move
from game import Game, NumpyEncoder
from mcts_agent import MCTS_Agent, MCTSNode
from model_manager import ModelManager, create_new_model
from random_network import RandomNetwork, PytorchAgentWrapper
from rl_agent import RLAgent
from noise import DirichletNoiseConfig
from train_data_handle import save_training_data
from train_network import train_network_steps, train_network_epochs
from self_play import generate_self_play_games, play_tournament_game
from train_data_handle import load_and_combine_data_by_count

In [2]:
SAVE_DIR = "omega_new\\silly_willy_models\\"
os.makedirs(SAVE_DIR, exist_ok=True)

DATA_SAVE_DIR = ".\\omega_new\\silly_willy_data\\" 
os.makedirs(DATA_SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Гиперпараметры
NUM_RES_BLOCKS = 15
NUM_SIMULATIONS = 400
MAX_MOVES_PER_GAME = 100
NUM_GAMES_TO_GENERATE = 100
GAMES_BATCH = 100
TRAINING_STEPS = 56
BATCH_SIZE = 256
LEARNING_RATE = 0.001

In [3]:
def create_agent_name() -> str:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"agent_{timestamp}.pth"
    return filename

import glob
def get_most_recent_filename(path: str) -> str:
    
    search_pattern = os.path.join(path, "agent_*.pth")
    filepaths = glob.glob(search_pattern)

    filepaths.sort(key=lambda x: -os.path.getmtime(x))
    if len(filepaths) > 0:
        return filepaths[0].split('\\')[-1]
    return None
get_most_recent_filename(SAVE_DIR)

'agent_20260306_200026.pth'

In [4]:

model_manager = ModelManager(
    RLAgent, 
    save_dir=SAVE_DIR, 
    device=DEVICE,
    NUM_RES_BLOCKS=NUM_RES_BLOCKS
)


In [5]:
RANDOM_GAMES = 1600
RANDOM_MAX_MOVES = 20

def get_or_create_recent_model():
    current_best_model_id = get_most_recent_filename(SAVE_DIR)
    if current_best_model_id == None:
        create_new_model(model_manager, create_agent_name())
        current_best_model_id = get_most_recent_filename(SAVE_DIR)
    best_model = model_manager.load_model_weights(current_best_model_id)
    return best_model

def create_random_games():
    best_model = get_or_create_recent_model()
    network = PytorchAgentWrapper(best_model, device=DEVICE)
    agent_rl = MCTS_Agent(network, num_simulations=NUM_SIMULATIONS, temperature=1.0)
        
    generate_self_play_games(games_to_generate=RANDOM_GAMES, games_batch=GAMES_BATCH, rl_agent = agent_rl, max_moves=MAX_MOVES_PER_GAME,
                                data_save_dir=DATA_SAVE_DIR, random_moves=RANDOM_MAX_MOVES, random_mcts=1600)

def initial_train():

    all_training_data = load_and_combine_data_by_count(DATA_SAVE_DIR, 100000)
    
    challenger_model = get_or_create_recent_model()
    optimizer = torch.optim.Adam(challenger_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4) 
    train_network_epochs(model=challenger_model, optimizer=optimizer, training_data=all_training_data, epochs=2, batch_size=256, device=DEVICE)
    
    challenger_id = create_agent_name()
    challenger_metadata = {'parent_version': 0}
    model_manager.save_checkpoint(challenger_model, optimizer, challenger_metadata, f"{challenger_id}")
    print(f"Новая модель-претендент '{challenger_id}' обучена и сохранена.")

In [6]:
current_best_model_id = get_most_recent_filename(SAVE_DIR)

random = RandomNetwork()
agent_random = MCTS_Agent(random, num_simulations=10, temperature=1.0)
#best_model = model_manager.load_model_weights(current_best_model_id)     
#network = PytorchAgentWrapper(best_model, device=DEVICE)
#agent_rl = MCTS_Agent(network, num_simulations=1200, temperature=1.0)

res = generate_self_play_games(games_to_generate=1, games_batch=1, rl_agent = agent_random, max_moves=300,
                            data_save_dir=DATA_SAVE_DIR, delay=0, play_until_end_chance=1, concede_value=-1, random_mcts=1000, random_moves=300)


ИГРА ОКОНЧЕНА! 300
 0|🟩🔴🟩⚫⚫🔴🔴🔴🔴🔴⚫
 1|🟩🔴⚫⚫🔴⚫🔴🔴⚫⚫⚫
 2|🔴🔴🔴⚫🔴🟩🔴⚫🔴🔴⚫
 3|🟩🟩🔴⚫🔴🟩🔴🟩🟩⚫⚫
 4|🟩🔴🔴⚫🟩🔴⚫🟩🔴🟩🟩
 5|🔴🔴⚫⚫🔴⚫⚫⚫🔴🟩🟩
 6|🔴⚫⚫⚫⚫⚫🔴⚫🔴🔴🟩
 7|🔴🟩⚫🔴🔴🟩🔴⚫⚫🟩🔴
 8|⚫🟩🟩⚫⚫⚫🟩🔴🔴⚫🟩
 9|⚫🔴⚫⚫🔴🔴🔴🔴🔴⚫🔴
10|🟩🔴🔴⚫⚫⚫⚫⚫🟩⚫🟩

Текущий игрок: 🔴 БЕЛЫЕ
Последовательных пасов: 0
*** ИГРА ОКОНЧЕНА ***

ПОДСЧЕТ ОЧКОВ (Китайские правила)
⚫ ЧЕРНЫЕ:
   Камни на доске: 46
   Территория:     0
   Коми:           0.0
   ИТОГО:          46.0

🔴 БЕЛЫЕ:
   Камни на доске: 48
   Территория:     5
   Коми:           0.5
   ИТОГО:          53.5

⚪ Нейтральные пункты: 22
🔴 БЕЛЫЕ ВЫИГРАЛИ с преимуществом 7.5 очков

Всего ходов: 101

Этап 0 сбора данных завершен. Собрано 101 состояний.
Попытка сохранить 101 записей в файл: .\omega_new\silly_willy_data\training_data_20260402_193610.json
✅ Данные успешно сохранены.


In [7]:
res.first_root.count_terminal_nodes_in_tree()


22

In [15]:
for i in range(50):
    current_best_model_id = get_most_recent_filename(SAVE_DIR)
    
    best_model = model_manager.load_model_weights(current_best_model_id)     
    network = PytorchAgentWrapper(best_model, device=DEVICE)
    agent_rl = MCTS_Agent(network, num_simulations=NUM_SIMULATIONS, temperature=1.0)
    
    generate_self_play_games(games_to_generate=NUM_GAMES_TO_GENERATE, games_batch=GAMES_BATCH, rl_agent = agent_rl, max_moves=MAX_MOVES_PER_GAME,
                            data_save_dir=DATA_SAVE_DIR, delay=0, play_until_end_chance=1, concede_value=-1)
    
    all_training_data = load_and_combine_data_by_count(DATA_SAVE_DIR, 100000)
    
    # Создаем новую модель-претендента на основе весов лучшей модели
    challenger_model = model_manager.load_model_weights(current_best_model_id)
    optimizer = torch.optim.Adam(challenger_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    
    # Обучаем (теперь все аргументы совпадают)
    train_network_steps(
        challenger_model,
        optimizer,
        all_training_data,
        steps=TRAINING_STEPS,
        batch_size=BATCH_SIZE,
        device=DEVICE 
    )
    
    # Сохраняем модель-претендента
    challenger_id = create_agent_name()
    challenger_metadata = {'parent_version': 0}
    model_manager.save_checkpoint(challenger_model, optimizer, challenger_metadata, f"{challenger_id}")
    print(f"Новая модель-претендент '{challenger_id}' обучена и сохранена.")


ХОД 4 | ИГРА 32/100 | ПРОШЛО 1722.557091474533c. | до конца False
 0|❎❎❎❎❎❎❎
 1|❎❎❎❎❎❎❎
 2|❎❎❎⚫❎❎❎
 3|❎❎❎⚫❎❎❎
 4|❎❎🔴❎❎❎❎
 5|❎❎❎❎❎❎❎
 6|❎❎❎❎❎❎❎

Текущий игрок: 🔴 БЕЛЫЕ
Последовательных пасов: 0

🔍 MCTS запущен (400 симуляций)...


KeyboardInterrupt: 

In [ ]:

os.makedirs(DATA_SAVE_DIR, exist_ok=True)


In [11]:
all_training_data = load_and_combine_data_by_count(DATA_SAVE_DIR, 15000)


Найдено 41 файлов. Цель: 15000 записей.
  ✅ Загружен: 'training_data_20260306_180952.json' | Записей в файле: 4023 | Всего собрано: 4023/15000
  ✅ Загружен: 'training_data_20260306_162615.json' | Записей в файле: 3198 | Всего собрано: 7221/15000
  ✅ Загружен: 'training_data_20260306_150116.json' | Записей в файле: 3825 | Всего собрано: 11046/15000
  ✅ Загружен: 'training_data_20260306_131835.json' | Записей в файле: 3627 | Всего собрано: 14673/15000
  ✅ Загружен: 'training_data_20260306_114210.json' | Записей в файле: 3395 | Всего собрано: 18068/15000
------------------------------------------------------------
Загрузка завершена.
Прочитано файлов: 5
Собрано записей: 18068 (Цель была: 15000)
------------------------------------------------------------


In [12]:
games = [i for i in all_training_data if i['turn'] == 0]
whites = [i for i in games if i['value'] != 1]
blacks = [i for i in games if i['value'] == 1]


len(whites), len(blacks)

(188, 312)

In [ ]:
current_best_model_id = get_most_recent_filename(SAVE_DIR)
        
best_model = model_manager.load_model_weights(current_best_model_id)     
network = PytorchAgentWrapper(best_model, device=DEVICE)
agent_rl = MCTS_Agent(network, num_simulations=NUM_SIMULATIONS, temperature=1.0)
        
generate_self_play_games(games_to_generate=NUM_GAMES_TO_GENERATE, games_batch=GAMES_BATCH, rl_agent = agent_rl, max_moves=MAX_MOVES_PER_GAME,
                         data_save_dir=DATA_SAVE_DIR, random_moves=MAX_MOVES_PER_GAME + 1)

In [10]:
from heatmaps import print_board_heatmaps_for_move
for i in range(50):
    print_board_heatmaps_for_move(
    move_index=i,
    history=0,
    states=all_training_data,
    thresholds=(5, 10, 15, 25, 50, 95),  # легко крутить границы
)            
    #0:5 5:10 10:15 15:25 25:50 40:95
    #🟢 🟡    🟠   🟤    🔴   🟣     🔵
    
    clear_output(wait=True)
    time.sleep(1)

KeyboardInterrupt: 

In [ ]:

MAX_MOVES_PER_GAME = 30

true_random = RandomNetwork()
true_random.randomise_value = True
true_random.randomise_score = False
true_random.value_modifier = 0.1

good = MCTS_Agent(network=true_random, num_simulations=2000, 
                  temperature=0, score_goal_mod = -10)
evil = MCTS_Agent(network=true_random, num_simulations=2, 
                  temperature=0, score_goal_mod = 0)

wins_challenger = 0
wins_best = 0

for i in range(TOURNAMENT_GAMES):
    print(f"\n--- Турнирная игра {i+1}/{TOURNAMENT_GAMES} ---")
    
    # Чередуем, кто играет черными, чтобы исключить преимущество первого хода
    if i % 2 == 0:
        game = play_tournament_game(good, evil, max_moves=MAX_MOVES_PER_GAME, 
                                    extra_text = f"ИГРА {i+1}/{TOURNAMENT_GAMES} | GOOD - ЧЕРНЫЕ | СЧЕТ - П{wins_challenger}/Ч{i - wins_challenger}")
        game.print_score()
        if game.get_winner() == game.board.BLACK:
            wins_challenger += 1
    else:
        game = play_tournament_game(evil, good, max_moves=MAX_MOVES_PER_GAME,
                                   extra_text = f"ИГРА {i+1}/{TOURNAMENT_GAMES} | GOOD - БЕЛЫЕ | СЧЕТ - П{wins_challenger}/Ч{i - wins_challenger}")
        game.print_score()
        if game.get_winner() == game.board.WHITE:
            wins_challenger += 1


    
    wins_best = TOURNAMENT_GAMES - wins_challenger
    
    print(f"Победитель партии: {game.get_winner()}. Текущий счет Претендента: {wins_challenger}/{i+1}")


print(f"\nРезультаты турнира: Претендент {wins_challenger} - {wins_best} Чемпион")